# PanTHERIA traits exploration

Exploratory analysis of the committed PanTHERIA WR05 source for the four candidate Kruger herbivores. The notebook uses the raw file for inspection; `scripts/build_traits.py` produces the normalized table used by the project.

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
data_path = Path("../data/raw/PanTHERIA_1-0_WR05_Aug2008.txt")
df = pd.read_table(data_path, sep="\t")

In [ ]:
df.head()

,MSW05_Order,MSW05_Family,MSW05_Genus,MSW05_Species,MSW05_Binomial,1-1_ActivityCycle,5-1_AdultBodyMass_g,8-1_AdultForearmLen_mm,13-1_AdultHeadBodyLen_mm,2-1_AgeatEyeOpening_d,...,26-6_GR_MinLong_dd,26-7_GR_MidRangeLong_dd,27-1_HuPopDen_Min_n/km2,27-2_HuPopDen_Mean_n/km2,27-3_HuPopDen_5p_n/km2,27-4_HuPopDen_Change,28-1_Precip_Mean_mm,28-2_Temp_Mean_01degC,30-1_AET_Mean_mm,30-2_PET_Mean_mm
0,Artiodactyla,Camelidae,Camelus,dromedarius,Camelus dromedarius,3.0,492714.47,-999.0,-999.00,-999.00,...,-999.00,-999.00,-999,-999.00,-999.0,-999.00,-999.00,-999.00,-999.00,-999.00
1,Carnivora,Canidae,Canis,adustus,Canis adustus,1.0,10392.49,-999.0,745.32,-999.00,...,-17.53,13.00,0,35.20,1.0,0.14,90.75,236.51,922.90,1534.40
2,Carnivora,Canidae,Canis,aureus,Canis aureus,2.0,9658.70,-999.0,827.53,7.50,...,-17.05,45.74,0,79.29,0.0,0.10,44.61,217.23,438.02,1358.98
3,Carnivora,Canidae,Canis,latrans,Canis latrans,2.0,11989.10,-999.0,872.39,11.94,...,-168.12,-117.60,0,27.27,0.0,0.06,53.03,58.18,503.02,728.37
4,Carnivora,Canidae,Canis,lupus,Canis lupus,2.0,31756.51,-999.0,1055.00,14.01,...,-171.84,3.90,0,37.87,0.0,0.04,34.79,4.82,313.33,561.11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5411,Rodentia,Muridae,Zyzomys,argurus,Zyzomys argurus,-999.0,40.42,-999.0,107.83,-999.00,...,114.33,131.09,0,1.10,0.0,0.02,62.33,256.75,692.93,1704.98
5412,Rodentia,Muridae,Zyzomys,maini,Zyzomys maini,-999.0,93.99,-999.0,-999.00,-999.00,...,131.45,132.66,0,0.17,0.0,0.00,90.76,265.30,877.90,1755.73
5413,Rodentia,Muridae,Zyzomys,palatilis,Zyzomys palatilis,-999.0,123.00,-999.0,-999.00,-999.00,...,136.72,137.08,0,0.00,0.0,-999.00,49.00,247.16,637.90,1638.67
5414,Rodentia,Muridae,Zyzomys,pedunculatus,Zyzomys pedunculatus,-999.0,100.00,-999.0,126.79,-999.00,...,130.16,132.97,0,0.09,0.0,0.25,21.64,215.72,291.82,1405.85


In [10]:
# PanTHERIA encodes missing values as -999; keep the raw df unchanged and
# create a cleaned view for analysis.
clean_df = df.replace(-999, pd.NA)

model_fields = {
    "adult_mass_kg": "5-1_AdultBodyMass_g",
    "gestation_days": "9-1_GestationLen_d",
    "interbirth_days": "14-1_InterbirthInterval_d",
    "longevity_months": "17-1_MaxLongevity_m",
    "maturity_days": "23-1_SexualMaturityAge_d",
    "litter_size": "15-1_LitterSize",
    "litters_per_year": "16-1_LittersPerYear",
}
targets = {
    "Zebra": "Equus quagga",
    "Wildebeest": "Connochaetes taurinus",
    "Waterbuck": "Kobus ellipsiprymnus",
    "Kudu": "Tragelaphus strepsiceros",
}

# Dataset-wide completeness for fields needed by the population model.
completeness = (
    clean_df[list(model_fields.values())].notna().mean().mul(100).round(1)
    .rename(index={value: key for key, value in model_fields.items()})
    .sort_values()
    .to_frame("complete_pct")
)
print(f"Dataset shape: {df.shape[0]:,} species x {df.shape[1]} fields")
display(completeness)

# Target-species trait extract, with human-readable units.
target_df = clean_df[clean_df["MSW05_Binomial"].isin(targets.values())].copy()
target_df["common_name"] = target_df["MSW05_Binomial"].map(
    {binomial: common for common, binomial in targets.items()}
)
target_summary = target_df[["common_name"] + list(model_fields.values())].rename(
    columns={value: key for key, value in model_fields.items()}
).set_index("common_name").sort_index()
target_summary["adult_mass_kg"] = target_summary["adult_mass_kg"] / 1000
target_summary["longevity_years"] = target_summary.pop("longevity_months") / 12
target_summary["maturity_years"] = target_summary.pop("maturity_days") / 365.25
target_summary["gestation_months"] = target_summary.pop("gestation_days") / 30.44
display(target_summary.round(2))

# Compare the model's gestation-only birth-rate proxy with interbirth data.
birth_rate = target_summary["gestation_months"].rdiv(12).rename("gestation_only_births_per_year")
interbirth_rate = (365.25 / target_summary["interbirth_days"]).rename("interbirth_births_per_year")
birth_rate_comparison = pd.concat([birth_rate, interbirth_rate], axis=1)
birth_rate_comparison["proxy_to_interbirth_ratio"] = (
    birth_rate_comparison.iloc[:, 0] / birth_rate_comparison.iloc[:, 1]
)
display(birth_rate_comparison.round(2))


Dataset shape: 5,416 species x 55 fields


,complete_pct
interbirth_days,12.8
litters_per_year,16.5
longevity_months,18.7
maturity_days,19.4
gestation_days,25.1
litter_size,46.2
adult_mass_kg,65.4


,adult_mass_kg,interbirth_days,litter_size,litters_per_year,longevity_years,maturity_years,gestation_months
common_name,,,,,,,
Kudu,206.05641,365.0,1.0,<NA>,23.0,1.821711,7.258213
Waterbuck,204.39348,365.0,1.0,<NA>,19.92,3.90538,8.869908
Wildebeest,198.61968,365.0,0.98,<NA>,21.5,2.304778,8.261827
Zebra,400.0,<NA>,<NA>,<NA>,20.0,<NA>,<NA>


,gestation_only_births_per_year,interbirth_births_per_year,proxy_to_interbirth_ratio
common_name,,,
Kudu,1.6533,1.000685,1.652168
Waterbuck,1.352889,1.000685,1.351963
Wildebeest,1.452463,1.000685,1.451469
Zebra,<NA>,<NA>,<NA>


### Initial findings

- PanTHERIA contains 5,416 species and 55 fields. The model-relevant fields are sparse across the full table, so filtering to the selected species and retaining `-999` as missing are essential.
- Adult mass and maximum longevity are available for all four candidates.
- Wildebeest, waterbuck, and kudu have gestation and interbirth values. Zebra has neither, and also lacks sexual maturity age, so its demographic parameters need another source before it can enter the model.
- For the three species with both measures, the current gestation-only birth-rate proxy is about 1.35 to 1.65 times the interbirth-based rate. This supports replacing the proxy with `InterbirthInterval_d` when the model is connected to the cleaned traits table.
- The table does not settle the waterbuck-versus-kudu choice; that decision also depends on which species has reliable Kruger census counts and appropriate stage-proportion assumptions.